# Assignment: Evaluate a Multimodal Model

**Unit:** Building & Evaluating Multimodal Pipelines | **Companion reading:** Evaluation, Benchmarks, and Safety
**Suggested time:** about 1 hour

---

## What is this assignment trying to do?

Every model card on Hugging Face lists benchmark scores. In the lesson you saw that a high score does not prove a model actually *looks* at its input. In this assignment you will test a small open vision-language model the way an ML team would before deciding whether to ship it.

1. **Part 1:** Score the model's free-form answers fairly.
2. **Part 2:** Run the model on a test of 30 pairs of real photos and measure its accuracy.
3. **Part 3:** Implement the **modality-reliance protocol** from the lesson (full, blind, mismatched, and counterfactual inputs) and compute the **reliance gap**, **change rate**, and **flip rate**.

### Where the test photos come from

The test set uses real photos from **NaturalBench**, a research benchmark built to catch models that answer without looking. Every question comes with **two similar photos that have different answers**, such as a red car and an orange car. The second photo is the *counterfactual*: if the model really looks, its answer should change between the two.

### How this assignment is graded

Your functions are **autograded**: the autograder runs them on its **own** test models, so it never needs your notebook's model or a GPU. That means:

- Keep every function **name and signature** exactly as given.
- Your functions must use the model only through the `ask_fn` argument they receive, never through a global model.

The two written reflections are also checked, and may be read by staff.

### How to work

- Replace every `...` with your own code.
- Write the reflections as Python strings in the cells provided.
- Run every cell top to bottom before submitting.

---
## Part 0: Setup

**Use a GPU.** In Colab: **Runtime → Change runtime type → T4 GPU**. Then run the next cells. Everything in this part is provided.

In [ ]:
# Colab: uncomment and run once if the model below fails to load, then restart the session
# !pip install -q -U transformers accelerate

import json
import os
import random
import string
import time

import numpy as np
from PIL import Image

SEED = 0

import matplotlib.pyplot as plt
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. This small model still runs on a CPU, just slowly; switch to a T4 if you can.")

### The test photos and the test set (PROVIDED)

The test photos come in a zip file, `multimodal_eval_images.zip` (about 5 MB). The next cell downloads it and unzips it into a folder called `multimodal_eval_images`. If the download fails, download the zip from the assignment page on the course website, upload it with the **Files** panel on the left side of Colab, and run the cell again.

`load_image(name)` opens a photo, and `blank_image()` returns a plain gray image. `TEST_SET` holds **30 questions in 5 categories** (6 each): `count`, `color`, `direction`, `state`, and `text`. Every question has exactly two possible answers, which most questions name, as in "What color is the car: red or orange?". Every item is a dictionary with:

- `id`, `category`, `question`, `answer`, and `choices` (the two valid answers),
- `image`: the file name of the original photo, and
- `cf_image` and `cf_answer`: the **counterfactual** photo, where the answer is different, and its correct answer.

`add_mismatches(items)` gives every item a `mismatch_image`: the photo of a different item in the same category whose answer is different.

In [ ]:
# Loading images (PROVIDED)
IMAGE_DIR = "multimodal_eval_images"


def load_image(name):
    """Open a test photo by its file name."""
    return Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")


def blank_image():
    """The 'blind' input: a plain gray image with nothing in it."""
    return Image.new("RGB", (384, 384), (128, 128, 128))


PROMPT_SUFFIX = " Answer with one word or number."
CONDITIONS = ["full", "blind", "mismatched", "counterfactual"]

# Which image the model sees under each condition
IMAGE_FOR = {
    "full": lambda item: load_image(item["image"]),
    "blind": lambda item: blank_image(),
    "mismatched": lambda item: load_image(item["mismatch_image"]),
    "counterfactual": lambda item: load_image(item["cf_image"]),
}

In [ ]:
# Download and unzip the test photos (PROVIDED)
import urllib.request
import zipfile

IMAGES_URL = ("https://raw.githubusercontent.com/bwsi-forge-ai-challenge/bwsi-forge-ai-challenge.github.io/"
              "main/Assignments/2_multimodal_%2B_advanced_models/multimodal_eval_images.zip")
if not os.path.isdir(IMAGE_DIR):
    if not os.path.exists("multimodal_eval_images.zip"):
        try:
            urllib.request.urlretrieve(IMAGES_URL, "multimodal_eval_images.zip")
        except Exception as exc:
            raise RuntimeError("Could not download the test photos. Download multimodal_eval_images.zip from the "
                               "assignment page, upload it in Colab's Files panel, and run this cell again.") from exc
    with zipfile.ZipFile("multimodal_eval_images.zip") as z:
        z.extractall(".")
print(len(os.listdir(IMAGE_DIR)), "files in", IMAGE_DIR)

In [ ]:
# The class test set (PROVIDED). Do not edit: everyone tests on the same 30 photo pairs.
TEST_SET = [
    {'id': 'count-0', 'category': 'count', 'question': 'How many apples are in the image: two or eight?',
     'answer': '2', 'choices': ['2', '8'], 'image': 'count-0.jpg', 'cf_image': 'count-0_cf.jpg', 'cf_answer': '8'},
    {'id': 'count-1', 'category': 'count', 'question': 'How many eggs are in the image: one or two?',
     'answer': '1', 'choices': ['1', '2'], 'image': 'count-1.jpg', 'cf_image': 'count-1_cf.jpg', 'cf_answer': '2'},
    {'id': 'count-2', 'category': 'count', 'question': 'How many cats are in the image: one or two?',
     'answer': '2', 'choices': ['1', '2'], 'image': 'count-2.jpg', 'cf_image': 'count-2_cf.jpg', 'cf_answer': '1'},
    {'id': 'count-3', 'category': 'count', 'question': 'How many cats are in the image: one or two?',
     'answer': '1', 'choices': ['1', '2'], 'image': 'count-3.jpg', 'cf_image': 'count-3_cf.jpg', 'cf_answer': '2'},
    {'id': 'count-4', 'category': 'count', 'question': 'How many traffic cones are in the image: one or two?',
     'answer': '1', 'choices': ['1', '2'], 'image': 'count-4.jpg', 'cf_image': 'count-4_cf.jpg', 'cf_answer': '2'},
    {'id': 'count-5', 'category': 'count', 'question': 'How many horses are in the image: two or eight?',
     'answer': '2', 'choices': ['2', '8'], 'image': 'count-5.jpg', 'cf_image': 'count-5_cf.jpg', 'cf_answer': '8'},
    {'id': 'color-0', 'category': 'color', 'question': 'What color is the car: red or orange?',
     'answer': 'red', 'choices': ['red', 'orange'], 'image': 'color-0.jpg', 'cf_image': 'color-0_cf.jpg', 'cf_answer': 'orange'},
    {'id': 'color-1', 'category': 'color', 'question': 'What color is the truck: white or black?',
     'answer': 'white', 'choices': ['white', 'black'], 'image': 'color-1.jpg', 'cf_image': 'color-1_cf.jpg', 'cf_answer': 'black'},
    {'id': 'color-2', 'category': 'color', 'question': 'What color is the bike: black or purple?',
     'answer': 'black', 'choices': ['black', 'purple'], 'image': 'color-2.jpg', 'cf_image': 'color-2_cf.jpg', 'cf_answer': 'purple'},
    {'id': 'color-3', 'category': 'color', 'question': 'What color is the sign: red or green?',
     'answer': 'red', 'choices': ['red', 'green'], 'image': 'color-3.jpg', 'cf_image': 'color-3_cf.jpg', 'cf_answer': 'green'},
    {'id': 'color-4', 'category': 'color', 'question': 'What color is the table: orange or red?',
     'answer': 'orange', 'choices': ['orange', 'red'], 'image': 'color-4.jpg', 'cf_image': 'color-4_cf.jpg', 'cf_answer': 'red'},
    {'id': 'color-5', 'category': 'color', 'question': "What color are the church's front doors: white or black?",
     'answer': 'white', 'choices': ['white', 'black'], 'image': 'color-5.jpg', 'cf_image': 'color-5_cf.jpg', 'cf_answer': 'black'},
    {'id': 'direction-0', 'category': 'direction', 'question': 'Is the heron facing left or right?',
     'answer': 'right', 'choices': ['left', 'right'], 'image': 'direction-0.jpg', 'cf_image': 'direction-0_cf.jpg', 'cf_answer': 'left'},
    {'id': 'direction-1', 'category': 'direction', 'question': 'Is the airplane facing left or right?',
     'answer': 'left', 'choices': ['left', 'right'], 'image': 'direction-1.jpg', 'cf_image': 'direction-1_cf.jpg', 'cf_answer': 'right'},
    {'id': 'direction-2', 'category': 'direction', 'question': 'Is the car facing left or right?',
     'answer': 'right', 'choices': ['left', 'right'], 'image': 'direction-2.jpg', 'cf_image': 'direction-2_cf.jpg', 'cf_answer': 'left'},
    {'id': 'direction-3', 'category': 'direction', 'question': 'Is the lizard facing left or right?',
     'answer': 'left', 'choices': ['left', 'right'], 'image': 'direction-3.jpg', 'cf_image': 'direction-3_cf.jpg', 'cf_answer': 'right'},
    {'id': 'direction-4', 'category': 'direction', 'question': 'Is the grasshopper facing left or right?',
     'answer': 'left', 'choices': ['left', 'right'], 'image': 'direction-4.jpg', 'cf_image': 'direction-4_cf.jpg', 'cf_answer': 'right'},
    {'id': 'direction-5', 'category': 'direction', 'question': 'Is the van facing to the left?',
     'answer': 'yes', 'choices': ['yes', 'no'], 'image': 'direction-5.jpg', 'cf_image': 'direction-5_cf.jpg', 'cf_answer': 'no'},
    {'id': 'state-0', 'category': 'state', 'question': 'Is the tiger standing or sitting?',
     'answer': 'standing', 'choices': ['standing', 'sitting'], 'image': 'state-0.jpg', 'cf_image': 'state-0_cf.jpg', 'cf_answer': 'sitting'},
    {'id': 'state-1', 'category': 'state', 'question': 'Is the dog sitting or lying down?',
     'answer': 'lying', 'choices': ['sitting', 'lying'], 'image': 'state-1.jpg', 'cf_image': 'state-1_cf.jpg', 'cf_answer': 'sitting'},
    {'id': 'state-2', 'category': 'state', 'question': 'Is the deer standing or lying down?',
     'answer': 'standing', 'choices': ['standing', 'lying'], 'image': 'state-2.jpg', 'cf_image': 'state-2_cf.jpg', 'cf_answer': 'lying'},
    {'id': 'state-3', 'category': 'state', 'question': 'Is the light turned on?',
     'answer': 'no', 'choices': ['yes', 'no'], 'image': 'state-3.jpg', 'cf_image': 'state-3_cf.jpg', 'cf_answer': 'yes'},
    {'id': 'state-4', 'category': 'state', 'question': 'Was this photo taken indoors or outdoors?',
     'answer': 'indoors', 'choices': ['indoors', 'outdoors'], 'image': 'state-4.jpg', 'cf_image': 'state-4_cf.jpg', 'cf_answer': 'outdoors'},
    {'id': 'state-5', 'category': 'state', 'question': "Are the butterfly's wings open or closed?",
     'answer': 'open', 'choices': ['open', 'closed'], 'image': 'state-5.jpg', 'cf_image': 'state-5_cf.jpg', 'cf_answer': 'closed'},
    {'id': 'text-0', 'category': 'text', 'question': 'Does the wall say "NO DUMPING"?',
     'answer': 'yes', 'choices': ['yes', 'no'], 'image': 'text-0.jpg', 'cf_image': 'text-0_cf.jpg', 'cf_answer': 'no'},
    {'id': 'text-1', 'category': 'text', 'question': 'Is there a sign that says "EXIT"?',
     'answer': 'no', 'choices': ['yes', 'no'], 'image': 'text-1.jpg', 'cf_image': 'text-1_cf.jpg', 'cf_answer': 'yes'},
    {'id': 'text-2', 'category': 'text', 'question': 'Is the word "WATER" shown in the image?',
     'answer': 'yes', 'choices': ['yes', 'no'], 'image': 'text-2.jpg', 'cf_image': 'text-2_cf.jpg', 'cf_answer': 'no'},
    {'id': 'text-3', 'category': 'text', 'question': 'Which word is on the sign: restore or winter?',
     'answer': 'restore', 'choices': ['restore', 'winter'], 'image': 'text-3.jpg', 'cf_image': 'text-3_cf.jpg', 'cf_answer': 'winter'},
    {'id': 'text-4', 'category': 'text', 'question': 'Is there a red "EXIT" sign in the image?',
     'answer': 'yes', 'choices': ['yes', 'no'], 'image': 'text-4.jpg', 'cf_image': 'text-4_cf.jpg', 'cf_answer': 'no'},
    {'id': 'text-5', 'category': 'text', 'question': 'Does the manhole cover say "WATER"?',
     'answer': 'no', 'choices': ['yes', 'no'], 'image': 'text-5.jpg', 'cf_image': 'text-5_cf.jpg', 'cf_answer': 'yes'},
]


def add_mismatches(items):
    """Give every item a 'mismatch_image': the photo of another item in the same
    category whose answer is different."""
    for k, item in enumerate(items):
        partner = next(o for o in items[k + 1:] + items[:k]
                       if o["category"] == item["category"] and o["answer"] != item["answer"])
        item["mismatch_image"] = partner["image"]


CATEGORIES = sorted({item["category"] for item in TEST_SET})

missing = [n for i in TEST_SET for n in (i["image"], i["cf_image"]) if not os.path.exists(os.path.join(IMAGE_DIR, n))]
assert not missing, f"missing photos {missing[:3]}: rerun the download cell"
print(len(TEST_SET), "items:", {c: sum(i["category"] == c for i in TEST_SET) for c in CATEGORIES})


def show_items(items, title=""):
    fig, axes = plt.subplots(2, len(items), figsize=(2.8 * len(items), 5.6))
    axes = np.array(axes).reshape(2, len(items))
    for col, item in enumerate(items):
        for row, (name, ans) in enumerate([(item["image"], item["answer"]), (item["cf_image"], item["cf_answer"])]):
            ax = axes[row, col]
            ax.imshow(load_image(name)); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"{item['id']}{' (cf)' if row else ''}\nanswer: {ans}", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()


show_items([next(i for i in TEST_SET if i["category"] == c) for c in CATEGORIES],
           "Top: original photos. Bottom: counterfactual photos.")

### The model (PROVIDED)

You will test **SmolVLM-256M**, a small open vision-language model that takes an image and a question and answers in text. The next cell loads it and defines `ask(image, question)`, which returns the model's answer as a string. Everything you write below receives a function like this as its `ask_fn` argument.

In [ ]:
# Load the model and define ask() (PROVIDED)
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
model.eval()


def ask(image, question):
    """Ask the model one question about one PIL image. Returns only the answer text."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)
    if DEVICE == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


demo = TEST_SET[0]
print(demo["question"], "->", repr(ask(load_image(demo["image"]), demo["question"] + PROMPT_SUFFIX)))

---
## Part 1: Scoring Free-Form Answers

The model answers in free text. It might say `"Red."`, `"The car is red"`, `"two"`, or `"I think it is red or orange"`. Comparing strings exactly would mark most of these wrong, which is the "many answers can be right" problem from the lesson.

`normalize(text)` is provided: it lowercases the text, replaces punctuation with spaces, turns number words like `two` into digits, and returns the list of words.

### TODO 1: `extract_answer(prediction, choices)`

Return the one choice from `choices` that the prediction names. If the prediction names none of them, or more than one (a hedge like "red or orange"), return `None`.

`is_correct` is provided and uses your `extract_answer`.

In [ ]:
NUMBER_WORDS = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
                "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}


def normalize(text):
    """Lowercase, replace punctuation with spaces, turn number words into digits, and split into words."""
    text = text.lower()
    for ch in string.punctuation:
        text = text.replace(ch, " ")
    return [NUMBER_WORDS.get(word, word) for word in text.split()]



def extract_answer(prediction, choices):
    """Return the one choice named in the prediction, or None if it names none or several."""
    words = normalize(prediction)
    found = ...
    return ...


def is_correct(prediction, answer, choices):
    return extract_answer(prediction, choices) == answer


def accuracy(results):
    return sum(r["correct"] for r in results) / len(results)


def category_accuracy(results):
    return {c: accuracy([r for r in results if r["category"] == c])
            for c in sorted({r["category"] for r in results})}

---
## Part 2: Running a Test Condition

### TODO 2: `run_condition(ask_fn, items, condition)`

`run_condition` asks `ask_fn` every question in `items` under one **condition** and returns a list of result dictionaries, one per item. The condition decides which image the model sees, and `IMAGE_FOR` already picks it for you:

| `condition` | Image the model sees |
|---|---|
| `"full"` | the original photo |
| `"blind"` | a blank gray image |
| `"mismatched"` | the photo of a different item whose answer is different |
| `"counterfactual"` | the counterfactual photo |

The loop, the timing, and the result dictionaries are provided. Fill in `target`: the answer that counts as correct under this condition. `accuracy` and `category_accuracy` are provided below the function.

In [ ]:
def run_condition(ask_fn, items, condition):
    """Ask every question under one condition; return one result dict per item."""
    results = []
    for item in items:
        image = IMAGE_FOR[condition](item)
        start = time.perf_counter()
        prediction = ask_fn(image, item["question"] + PROMPT_SUFFIX)
        seconds = time.perf_counter() - start
        target = ...
        results.append({
            "id": item["id"], "category": item["category"], "prediction": prediction,
            "extracted": extract_answer(prediction, item["choices"]),
            "correct": is_correct(prediction, target, item["choices"]), "seconds": seconds,
        })
    return results

In [ ]:
# Run the model on the test set under the full condition (PROVIDED)
full = run_condition(ask, TEST_SET, "full")
print(f"Full accuracy: {accuracy(full):.1%}")
print(f"Median latency per question: {np.median([r['seconds'] for r in full]):.2f} s")
for c, acc in category_accuracy(full).items():
    print(f"  {c:10s} {acc:.0%}")
print("\nWrong answers:")
for r in full:
    if not r["correct"]:
        item = next(i for i in TEST_SET if i["id"] == r["id"])
        print(f"  {r['id']:12s} expected {item['answer']!r:8s} model said {r['prediction']!r}")

### Part 2 reflection

Which category was the model best at, and which was it worst at? Pick **one wrong answer** from the list above, look at its photo (for example `display(load_image(TEST_SET[0]["image"]))`), and explain what you think went wrong. Was it a perception error, or did the model give an answer the scorer could not read? Write at least 30 words.

In [ ]:
part2_reflection = """
TODO: your answer here
"""

---
## Part 3: Does the Model Actually Look?

Full accuracy alone cannot tell grounded answers from good guesses. The lesson's **modality-reliance protocol** adds three more conditions:

- **Blind:** a plain gray image. A grounded model should do much worse.
- **Mismatched:** the image from a *different* item whose answer is different. A grounded model should change its answer.
- **Counterfactual:** the minimally edited image. A grounded model should switch to the new correct answer.

### TODO 3: The three reliance numbers

Using the definitions from the lesson, write:

1. `reliance_gap(full, blind)`: accuracy on the full run minus accuracy on the blind run.
2. `change_rate(full, mismatched)`: the fraction of items whose **extracted** answer differs between the full and mismatched runs.
3. `flip_rate(counterfactual)`: the fraction of counterfactual items the model got right.

Each argument is a list of result dictionaries from `run_condition`, in the same order, so item *i* of one list matches item *i* of another.

In [ ]:
def reliance_gap(full, blind):
    """Accuracy on the full run minus accuracy on the blind run."""
    return ...


def change_rate(full, mismatched):
    """Fraction of items whose extracted answer differs between the full and mismatched runs."""
    return ...


def flip_rate(counterfactual):
    """Fraction of counterfactual items the model answered correctly."""
    return ...

### TODO 4: `evaluate_model(ask_fn, items)`

`evaluate_model` runs the whole protocol on any model and any list of items. The report is provided; fill in `results`: a dictionary that maps each condition name in `CONDITIONS` to the list of results from `run_condition` for that condition.

In [ ]:
def evaluate_model(ask_fn, items):
    """Run all four conditions on `items` and return a report dictionary."""
    add_mismatches(items)
    results = ...
    return {
        "full_accuracy": accuracy(results["full"]),
        "blind_accuracy": accuracy(results["blind"]),
        "reliance_gap": reliance_gap(results["full"], results["blind"]),
        "change_rate": change_rate(results["full"], results["mismatched"]),
        "flip_rate": flip_rate(results["counterfactual"]),
        "median_latency_s": float(np.median([r["seconds"] for r in results["full"]])),
        "category_accuracy": category_accuracy(results["full"]),
        "results": results,
    }

In [ ]:
# Evaluate the model with the full protocol (PROVIDED)
REPORT = evaluate_model(ask, TEST_SET)
for k in ["full_accuracy", "blind_accuracy", "reliance_gap", "change_rate", "flip_rate"]:
    print(f"{k:15s} {REPORT[k]:.1%}")
print(f"median latency  {REPORT['median_latency_s']:.2f} s")

x = np.arange(len(CATEGORIES))
plt.figure(figsize=(7, 3.2))
for k, name in enumerate(["full", "blind", "counterfactual"]):
    accs = category_accuracy(REPORT["results"][name])
    plt.bar(x + (k - 1) * 0.27, [accs[c] for c in CATEGORIES], width=0.27, label=name)
plt.xticks(x, CATEGORIES); plt.ylabel("accuracy"); plt.ylim(0, 1.05); plt.legend(fontsize=8)
plt.title(MODEL_ID); plt.tight_layout(); plt.show()

### Part 3 reflection

Compare your numbers with **Model A** and **Model B** in the lesson's worked example. Is the model closer to A (scores well but mostly guesses) or B (clearly uses the image)? Point to the specific numbers that convince you. Then look at the **blind** predictions (`[r["prediction"] for r in REPORT["results"]["blind"]]`): what does the model say when there is nothing to see? Does it admit it cannot tell, or does it answer anyway? Write at least 30 words.

In [ ]:
part3_reflection = """
TODO: your answer here
"""

---
## Submitting

1. Make sure every `TODO` is complete and both reflections (`part2_reflection`, `part3_reflection`) are written.
2. Run all cells from top to bottom (**Runtime → Run all** in Colab).
3. Download the notebook as a Python file (**File → Download → Download .py**) and rename it `firstname_lastname_model_eval.py`.
4. Submit that `.py` file on Gradescope (see the assignment page on the course website for the link).